# Smart Microscopy v2

**Config** — imports and all parameters.  
**Connect** — connect to LAS X.  
**Step 1** — set stage limits, strip template, enforce z-galvo.  
**Step 2** — draw the scan field in Navigator Expert, then run: reads and visualises tile positions.  
**Step 3** — place focus point markers on the scan field in Navigator Expert, then run: autofocus at each marker, fit Z plane.  
**Step 4** — acquire all tiles with interpolated Z.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path("controller/vendor/leica").resolve()))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import lasx
from lasx.scanning_templates import TEMPLATE_BASE, TEMPLATE_XML, STRIPPED_XML
from LasxApi import PYLICamApiConnector as lasxApi

# ── Stage limits (um) ────────────────────────────────────────────────
STAGE_X_MIN_UM   = None   # set to restrict XY travel to the sample area
STAGE_X_MAX_UM   = None   # leave as None to auto-derive from scan field in Step 2
STAGE_Y_MIN_UM   = None
STAGE_Y_MAX_UM   = None
LIMIT_MARGIN_UM  = 500    # XY padding when limits are auto-derived from scan field
Z_GALVO_MIN      = -200   # hardware-specific
Z_GALVO_MAX      =  200
Z_WIDE_MIN       = -5000
Z_WIDE_MAX       =  5000

# ── Jobs ─────────────────────────────────────────────────────────────
AF_JOB           = "AF Job"
ACQUISITION_JOB  = "Overview"

# ── Autofocus ────────────────────────────────────────────────────────
RESTORE_AFTER_AF = True

In [ ]:
client = lasxApi.LasxApiClientPyModel
client.Connect("PythonClient")
client.PyApiClient.DelayInMilliseconds = 300
mode = client.PyApiSetApiInterfaceToUse.Model.ApiInterfaceToUse
client.PyApiSetApiInterfaceToUse.Model.ApiInterfaceToUse = (
    type(mode).Only_the_CAM_interface_is_used
)
assert lasx.ping(client), "LAS X not responding"

templates_dir = lasx.find_scanning_templates_dir()
print(f"Connected — templates: {templates_dir}")

## Step 1 — Stage limits

Set `STAGE_X/Y_MIN/MAX_UM` in the config cell, then run this cell.  
Strips the template and enforces z-galvo on all jobs.

In [ ]:
xy_limit_values = [STAGE_X_MIN_UM, STAGE_X_MAX_UM, STAGE_Y_MIN_UM, STAGE_Y_MAX_UM]
if all(v is not None for v in xy_limit_values):
    lasx.set_stage_limits(
        x_min=STAGE_X_MIN_UM,
        x_max=STAGE_X_MAX_UM,
        y_min=STAGE_Y_MIN_UM,
        y_max=STAGE_Y_MAX_UM,
        z_galvo_min=Z_GALVO_MIN,
        z_galvo_max=Z_GALVO_MAX,
        z_wide_min=Z_WIDE_MIN,
        z_wide_max=Z_WIDE_MAX,
    )
    lim = lasx.get_stage_limits()
    print(f"Stage limits set:")
    print(f"  X: {lim['x_min']:.0f} \u2013 {lim['x_max']:.0f} um")
    print(f"  Y: {lim['y_min']:.0f} \u2013 {lim['y_max']:.0f} um")
else:
    print("Stage limits: not set (STAGE_X/Y_MIN/MAX_UM are None).")
    print("  \u2192 Limits will be derived from the scan field in Step 2.")

strip_result = lasx.strip_template(client)
assert strip_result, "Strip failed"
print(f"\nTemplate stripped — draw a new scan field in Navigator Expert before running Step 2.")

def _set_z_galvo(lrp_path):
    parsed = lasx.parse_lrp(lrp_path)
    for name in parsed["jobs"]:
        lasx.lrp_set_z_use_mode(lrp_path, "z-galvo", name)

def _verify_z_galvo(lrp_path):
    parsed = lasx.parse_lrp(lrp_path)
    return all(
        lasx.lrp_verify_z_use_mode(lrp_path, "z-galvo", n)
        for n in parsed["jobs"]
    )

result = lasx.apply_lrp_change(client, STRIPPED_XML, _set_z_galvo, verify_fn=_verify_z_galvo)
assert result and result["success"], "z-galvo enforcement failed"
print("z-galvo enforced on all jobs")

## Step 2 — Scan field

Draw the scanning field in **Navigator Expert**, then run this cell.

In [ ]:
lasx.save_experiment(client, TEMPLATE_XML, templates_dir, timeout=60)
template_data = lasx.parse_template_positions(templates_dir, TEMPLATE_BASE, client=client)

tile_positions = template_data.get("acquisition_positions", {})
if not tile_positions and template_data.get("geometries"):
    from lasx.scanning_template_parsers import _tile_size_from_image_size_str
    settings = lasx.get_job_settings(client, ACQUISITION_JOB)
    tile_size_um = None
    if settings:
        tile_size_um = _tile_size_from_image_size_str(settings.get("imageSize", ""))
    if tile_size_um:
        template_data = lasx.synthesize_tiles(template_data, tile_size_um, job_name=ACQUISITION_JOB)
        tile_positions = template_data["acquisition_positions"]
        n_synth = sum(len(r["positions"]) for r in tile_positions.values())
        print(f"Synthesized {n_synth} tiles from geometries (tile size {tile_size_um:.1f} um)")
    else:
        print("WARNING: Cannot determine tile size — no tiles generated")

n_tiles = sum(len(r["positions"]) for r in tile_positions.values())
print(f"Scan field: {len(tile_positions)} region(s), {n_tiles} tile(s)")
for rid, region in tile_positions.items():
    print(f"  Region {rid}: {region['job_name']}  "
          f"{region.get('num_rows', '?')}\u00d7{region.get('num_cols', '?')}  "
          f"tile={region.get('tile_size_um', '?')} um")

# ── Stage limits: explicit config or derived from scan field ─────────
tile_centers_x = [p["x_um"] for r in tile_positions.values() for p in r["positions"]]
tile_centers_y = [p["y_um"] for r in tile_positions.values() for p in r["positions"]]

xy_config = [STAGE_X_MIN_UM, STAGE_X_MAX_UM, STAGE_Y_MIN_UM, STAGE_Y_MAX_UM]
if all(v is not None for v in xy_config):
    lim = lasx.get_stage_limits()
    print(f"\nStage limits (from config):")
elif tile_centers_x:
    ts_half = max(
        (r.get("tile_size_um") or 0) for r in tile_positions.values()
    ) / 2
    lasx.set_stage_limits(
        x_min=min(tile_centers_x) - ts_half - LIMIT_MARGIN_UM,
        x_max=max(tile_centers_x) + ts_half + LIMIT_MARGIN_UM,
        y_min=min(tile_centers_y) - ts_half - LIMIT_MARGIN_UM,
        y_max=max(tile_centers_y) + ts_half + LIMIT_MARGIN_UM,
        z_galvo_min=Z_GALVO_MIN,
        z_galvo_max=Z_GALVO_MAX,
        z_wide_min=Z_WIDE_MIN,
        z_wide_max=Z_WIDE_MAX,
    )
    lim = lasx.get_stage_limits()
    print(f"\nStage limits (derived from scan field + {LIMIT_MARGIN_UM} um margin):")
else:
    lim = None

if lim:
    print(f"  X: {lim['x_min']:.0f} \u2013 {lim['x_max']:.0f} um")
    print(f"  Y: {lim['y_min']:.0f} \u2013 {lim['y_max']:.0f} um")

# ── Visualise ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor("white")
ax.set_facecolor("#f5f5f8")

all_x, all_y = [], []
viz_colors = template_data.get("visualization_data", {}).get("tile_colors", {})
job_color_map = {
    region["job_name"]: tuple(viz_colors[region["job_name"]])
    for region in tile_positions.values()
    if region["job_name"] in viz_colors
}
legend_jobs = set()

for rid, region in tile_positions.items():
    jn = region["job_name"]
    ts = region.get("tile_size_um")
    if ts is None:
        continue
    half = ts / 2
    rgba = job_color_map.get(jn, (0.78, 0.78, 0.78, 1.0))
    face = (rgba[0], rgba[1], rgba[2], 0.25)
    edge = (rgba[0], rgba[1], rgba[2], 0.80)
    for pos in region["positions"]:
        cx, cy = pos["x_um"], pos["y_um"]
        ax.add_patch(patches.Rectangle(
            (cx - half, cy - half), ts, ts,
            linewidth=0.6, edgecolor=edge, facecolor=face, zorder=2,
        ))
        all_x.extend([cx - half, cx + half])
        all_y.extend([cy - half, cy + half])
    if jn not in legend_jobs:
        label = "No job assigned" if jn == "(unassigned)" else jn
        ax.plot([], [], "s", color=(rgba[0], rgba[1], rgba[2], 0.6), markersize=8, label=label)
        legend_jobs.add(jn)

if lim:
    ax.add_patch(patches.Rectangle(
        (lim["x_min"], lim["y_min"]),
        lim["x_max"] - lim["x_min"],
        lim["y_max"] - lim["y_min"],
        linewidth=1.0, edgecolor="#aaaaaa", facecolor="none",
        linestyle=(0, (5, 4)), zorder=1,
    ))
    ax.plot([], [], ls=(0, (5, 4)), color="#aaaaaa", linewidth=1.0, label="Stage limits")
    all_x.extend([lim["x_min"], lim["x_max"]])
    all_y.extend([lim["y_min"], lim["y_max"]])

if all_x:
    span = max(max(all_x) - min(all_x), max(all_y) - min(all_y))
    cross = span * 0.006
    circle_r = cross * 0.6
    for fp_list, color, label in [
        (template_data.get("focus_points", []),     "#4EB8B8", "Focus points"),
        (template_data.get("autofocus_points", []), "#5CBF5C", "AutoFocus points"),
    ]:
        for fp in fp_list:
            fx, fy = fp["x_um"], fp["y_um"]
            ax.plot([fx - cross, fx + cross], [fy, fy], "-", color=color, linewidth=0.8, zorder=10)
            ax.plot([fx, fx], [fy - cross, fy + cross], "-", color=color, linewidth=0.8, zorder=10)
            ax.add_patch(patches.Circle((fx, fy), circle_r,
                linewidth=0.8, edgecolor=color, facecolor="none", zorder=11))
            all_x.append(fx); all_y.append(fy)
        if fp_list:
            ax.plot([], [], "+", color=color, markersize=10, markeredgewidth=1.8, label=label)

    pad = span * 0.05
    ax.set_xlim(min(all_x) - pad, max(all_x) + pad)
    ax.set_ylim(min(all_y) - pad, max(all_y) + pad)

ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xticks([]); ax.set_yticks([])
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(0.8); spine.set_edgecolor("#cccccc")
ax.set_title("Scan Field", fontsize=13, fontweight="bold", color="#222222", pad=12)
ax.legend(loc="upper right", fontsize=9, facecolor="white", edgecolor="#cccccc", labelcolor="#444444")
plt.tight_layout()
plt.show()

## Step 3 — Autofocus

Place **point markers** on the scan field in **Navigator Expert** (focus reference positions), then run this cell.  
Runs the AF job at each marker, reads back Z, fits a plane.

In [ ]:
lasx.save_experiment(client, TEMPLATE_XML, templates_dir, timeout=60)
_af_data = lasx.parse_template_positions(templates_dir, TEMPLATE_BASE, client=client)

focus_positions = (
    _af_data.get("focus_points", []) or
    _af_data.get("autofocus_points", [])
)
if not focus_positions:
    raise RuntimeError(
        "No focus points found.\n"
        "Add focus or autofocus positions on the scan field in Navigator Expert, "
        "then re-run this cell."
    )

print(f"Focus positions ({len(focus_positions)}):")
for i, fp in enumerate(focus_positions):
    print(f"  {i + 1}. x={fp['x_um']:.1f}  y={fp['y_um']:.1f} um")

lasx.strip_template(client)
lasx.select_job(client, AF_JOB)

measured_z = []
for i, fp in enumerate(focus_positions):
    print(f"\n[{i + 1}/{len(focus_positions)}] x={fp['x_um']:.0f}  y={fp['y_um']:.0f}",
          end="", flush=True)
    lasx.move_xy(client, fp["x_um"], fp["y_um"])
    lasx.acquire(client, AF_JOB)
    settings = lasx.get_job_settings(client, AF_JOB)
    ch = lasx.make_changeable_copy(settings)
    z_um = ch["zPosition"]["z-galvo"]
    measured_z.append({**fp, "z_um": z_um})
    print(f"  z={z_um:.2f} um")

if RESTORE_AFTER_AF:
    lasx.restore_template(client)

xs = np.array([m["x_um"] for m in measured_z])
ys = np.array([m["y_um"] for m in measured_z])
zs = np.array([m["z_um"] for m in measured_z])
A = np.column_stack([xs, ys, np.ones(len(measured_z))])
plane_coeffs, *_ = np.linalg.lstsq(A, zs, rcond=None)

def interpolate_z(x, y):
    return plane_coeffs[0] * x + plane_coeffs[1] * y + plane_coeffs[2]

residuals = zs - np.array([interpolate_z(x, y) for x, y in zip(xs, ys)])
print(f"\nFocus plane:")
print(f"  Z range:      {zs.max() - zs.min():.2f} um")
print(f"  Tilt X:       {np.degrees(np.arctan(plane_coeffs[0])):+.4f} deg")
print(f"  Tilt Y:       {np.degrees(np.arctan(plane_coeffs[1])):+.4f} deg")
print(f"  Max residual: {np.max(np.abs(residuals)):.3f} um")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sc = ax.scatter(xs, ys, c=zs, cmap="coolwarm", s=200, edgecolors="k")
for m in measured_z:
    ax.annotate(f"{m['z_um']:.2f}", (m["x_um"], m["y_um"]),
                textcoords="offset points", xytext=(8, 8), fontsize=8)
plt.colorbar(sc, ax=ax, label="Z (um)")
ax.set_xlabel("X (um)"); ax.set_ylabel("Y (um)")
ax.set_title("Measured Focus Points")
ax.set_aspect("equal"); ax.invert_yaxis()

ax = axes[1]
tile_x = [p["x_um"] for r in tile_positions.values() for p in r["positions"]]
tile_y = [p["y_um"] for r in tile_positions.values() for p in r["positions"]]
if tile_x:
    margin = 50
    xi = np.linspace(min(tile_x) - margin, max(tile_x) + margin, 100)
    yi = np.linspace(min(tile_y) - margin, max(tile_y) + margin, 100)
    Xi, Yi = np.meshgrid(xi, yi)
    cf = ax.contourf(Xi, Yi, interpolate_z(Xi, Yi), levels=20, cmap="coolwarm")
    plt.colorbar(cf, ax=ax, label="Z (um)")
    ax.scatter(xs, ys, c="k", marker="*", s=100, zorder=5, label="Focus pts")
    ax.plot(tile_x, tile_y, ".", color="gray", markersize=2, label="Tiles")
    ax.legend(fontsize=8)
ax.set_xlabel("X (um)"); ax.set_ylabel("Y (um)")
ax.set_title("Interpolated Z Surface")
ax.set_aspect("equal"); ax.invert_yaxis()

plt.tight_layout()
plt.show()

## Step 4 — Acquire

Strips the template, acquires at every tile position with interpolated Z from the focus plane.

In [ ]:
lasx.strip_template(client)
lasx.select_job(client, ACQUISITION_JOB)

sequence = []
for rid, region in sorted(tile_positions.items(), key=lambda r: int(r[0])):
    rows = {}
    for p in region["positions"]:
        rows.setdefault(p["row"], []).append(p)
    for row_idx in sorted(rows):
        row_tiles = sorted(rows[row_idx], key=lambda p: p["col"])
        if row_idx % 2 == 1:
            row_tiles = row_tiles[::-1]
        for p in row_tiles:
            sequence.append({
                "region": rid,
                "x_um": p["x_um"],
                "y_um": p["y_um"],
                "z_um": interpolate_z(p["x_um"], p["y_um"]),
            })

print(f"Acquiring {len(sequence)} positions with '{ACQUISITION_JOB}'\n")

results = []
for i, pos in enumerate(sequence):
    print(f"[{i + 1}/{len(sequence)}] R{pos['region']}  "
          f"x={pos['x_um']:.0f}  y={pos['y_um']:.0f}  "
          f"z={pos['z_um']:.2f}", end="", flush=True)
    lasx.move_xy(client, pos["x_um"], pos["y_um"])
    lasx.move_z(client, ACQUISITION_JOB, pos["z_um"], z_mode="galvo")
    result = lasx.acquire(client, ACQUISITION_JOB)
    results.append({**pos, "success": result["success"]})
    elapsed = result["timing"]["total_s"] if result["success"] else 0
    status = "OK" if result["success"] else "FAIL"
    print(f"  {status} ({elapsed:.1f}s)")

ok = sum(1 for r in results if r["success"])
print(f"\nDone: {ok}/{len(results)} successful")